In [ ]:
import os
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy.optimize import curve_fit

from pollen_worker.pollen_utils import get_clients_population_dict

# Change font to respect submission requirements
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = "Times New Roman"
plt.rcParams['font.size'] = 14
plt.rcParams['font.weight'] = 'normal'

# srun -w mauao -c 5 --partition=interactive --pty bash
# srun -w ngongotaha -c 5 --partition=interactive --pty bash
# poetry shell
# jupyter server --no-browser --port=8889
# ssh -L 8889:localhost:8889 ls985@mauao
# ssh -L 8889:localhost:8889 ls985@ngongotaha

In [ ]:
# root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/2023-10-09/15-42-18", "shakespeare_memory", 10 # Shakespeare lb
# root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/2023-10-09/15-42-21", "shakespeare_memory", 10 # Shakespeare llb
# root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/2023-10-09/15-42-27", "openimage", 20 # Openimage lb
root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/2023-10-09/15-48-41", "openimage", 20 # Openimage llb
# root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/", "google_speech", 20 # Google Speech
# root_dir, dataset, batch_size = "/nfs-share/ls985/pollen_worker/outputs/", "reddit", 20 # Reddit
CLIENTS_TRAINING_STATS = f"{root_dir}/clients_training_stats.parquet"
# GPU_STATS = f"{root_dir}/gpu_stats.parquet"

In [ ]:
ctd = pq.read_table(CLIENTS_TRAINING_STATS).to_pandas()
# gs = pq.read_table(GPU_STATS).to_pandas()

In [ ]:
ctd

In [ ]:
# gs

In [ ]:
# gs.columns

In [ ]:
cid_samples_dict = get_clients_population_dict(
    name=dataset,
    batch_size=1,
    seed=1337
)

In [ ]:
# gs.plot(x=" timestamp", y=" utilization.gpu [%]", figsize=(5, 5))

In [ ]:
# gs.plot(x=" timestamp", y=" clocks.current.sm [MHz]", figsize=(5, 5))

In [ ]:
ctd['n_samples'] = ctd['cid'].apply(lambda x: cid_samples_dict[x])
ctd['n_batches'] = ctd['n_samples'] // batch_size

In [ ]:
ctd['delta_t'] = ctd['end_time'] - ctd['start_time']
ctd['delta_t_s'] = ctd['delta_t'].apply(lambda x: x * 1e-9)

In [ ]:

list_of_nodes = list(ctd.node.unique())
list_of_batches = list(ctd.n_batches.unique())
list_of_batches.sort()
ctd['is_outlier'] = False
for node in list_of_nodes:
    quantiles = dict()
    for bs in list_of_batches:
        quantiles[bs] = ctd[(ctd.n_batches == bs) & (ctd.node == node)]['delta_t_s'].quantile(0.66)
    ctd['is_outlier'] = ctd.apply(lambda x: (x['delta_t_s'] > quantiles[x['n_batches']]) if (x['node'] == node) else x['is_outlier'], axis=1)

In [ ]:
ctd[ctd["is_outlier"] == True]

In [ ]:
plt.figure(figsize=(15, 5))
sns.violinplot(
    data=ctd[ctd["is_outlier"] == False],
    x="n_batches",
    y="delta_t_s",
)
plt.ylabel("Time [s]")
plt.xlabel("Number of batches")
plt.title(f"{dataset.upper()} - violin plot (w/o outliers)")
plt.grid(axis='y')
# plt.savefig(f"{dataset}_violinplot_wo_outliers.pdf", format="pdf", dpi=800, bbox_inches='tight')
plt.show()

plt.figure(figsize=(15, 5))
sns.violinplot(
    data=ctd,
    x="n_batches",
    y="delta_t_s",
)
plt.ylabel("Time [s]")
plt.xlabel("Number of batches")
plt.title(f"{dataset.upper()} - violin plot (w/ outliers)")
plt.grid(axis='y')
# plt.savefig(f"{dataset}_violinplot_w_outliers.pdf", format="pdf", dpi=800, bbox_inches='tight')
plt.show()

In [ ]:
df = ctd[ctd["is_outlier"] == True]
x_data_out = df["n_batches"].values
y_data_out = df["delta_t"].values * 1e-9
df = ctd[ctd["is_outlier"] == False]
x_data = df["n_batches"].values
y_data = df["delta_t"].values * 1e-9
model = lambda x, A, B, offset:  offset+A*np.log(x+B)
popt, pcov = curve_fit(
    model,
    x_data,
    y_data,
    p0=[0,0,0]
)
df = ctd
x_data_all = df["n_batches"].values
y_data_all = df["delta_t"].values * 1e-9
popt_all, pcov_all = curve_fit(
    model,
    x_data_all,
    y_data_all,
    p0=[0,0,0]
)

plt.figure(figsize=(10, 5))
x = np.linspace(x_data.min(),x_data.max(),250)
plt.plot(x, model(x,*popt), label="Fit curve", color="red")
plt.plot(x, model(x,*popt_all), label="Fit curve w/ outliers", color="orange")
plt.scatter(x_data, y_data, label="Data", color="blue")
plt.scatter(x_data_out, y_data_out, label="Outliers", color="cyan")
plt.ylabel("Time [s]")
plt.xlabel("Number of batches")
plt.legend()
plt.title(f"{dataset.upper()} - clients training time (w/ outliers)")
plt.grid()
# plt.savefig(f"{dataset}_ctt_data_fit_w_outliers.pdf", format="pdf", dpi=800, bbox_inches='tight')

plt.figure(figsize=(10, 5))
x = np.linspace(x_data.min(),x_data.max(),250)
plt.plot(x, model(x,*popt), label="Fit curve", color="red")
plt.plot(x, model(x,*popt_all), label="Fit curve w/ outliers", color="orange")
plt.scatter(x_data, y_data, label="Data", color="blue")
plt.ylabel("Time [s]")
plt.xlabel("Number of batches")
plt.legend()
plt.title(f"{dataset.upper()} - clients training time (w/o outliers)")
plt.grid()
# plt.savefig(f"{dataset}_ctt_data_fit_wo_outliers.pdf", format="pdf", dpi=800, bbox_inches='tight')